# Final Workshop — Orders Medallion Pipeline

**Databricks Fundamentals | Day 1 | ~50 min**

---

## Scenario

You are a data engineer at a retail company. Your manager needs a **fully automated pipeline** that turns raw JSON order files into a daily revenue report — ready for the analytics team by 06:00 every morning.

You will implement the pipeline as three independent notebooks following the **Medallion architecture**, then wire them together into a Databricks Workflow Job.

```
[Volume]  orders_batch.json
      |
      v  Task 01 — lab_task_01_bronze
      |  Read JSON + add metadata -> bronze.lab_orders
      |
      v  Task 02 — lab_task_02_silver
      |  Cast * enrich * filter -> silver.lab_orders
      |
      v  Task 03 — lab_task_03_gold
         Aggregate by date -> gold.lab_daily_orders
```

---

## Layer responsibilities

| Layer | Table | Rule |
|-------|-------|------|
| **Bronze** | `bronze.lab_orders` | Raw data — only metadata columns added |
| **Silver** | `silver.lab_orders` | Typed, enriched, filtered |
| **Gold** | `gold.lab_daily_orders` | Aggregated — one row per day |

---

## Workshop files

| Notebook | Layer | What you build |
|----------|-------|----------------|
| `tasks/lab_task_01_bronze` | Bronze | Raw JSON -> Delta |
| `tasks/lab_task_02_silver` | Silver | Casting, enrichment, filtering |
| `tasks/lab_task_03_gold`   | Gold   | Daily revenue aggregation |

---

## Step 0 — Verify setup


In [ ]:
%run ../../setup/00_setup
print(f"Catalog  : {CATALOG}")
print(f"Bronze   : {BRONZE_SCHEMA}")
print(f"Silver   : {SILVER_SCHEMA}")
print(f"Gold     : {GOLD_SCHEMA}")
print(f"Dataset  : {DATASET_PATH}")

---

## Step 1 — Complete the task notebooks

Open and complete them **in order**. Each notebook:
- starts with `%run ../../setup/00_setup`
- has `dbutils.widgets` — run as-is
- has `# YOUR CODE HERE` cells to fill in
- ends with `dbutils.notebook.exit(json)` returning status

---

## Step 2 — Smoke test

Run this cell after completing all three task notebooks.


In [ ]:
tables = [
    (f"{CATALOG}.{BRONZE_SCHEMA}.lab_orders",     "bronze"),
    (f"{CATALOG}.{SILVER_SCHEMA}.lab_orders",     "silver"),
    (f"{CATALOG}.{GOLD_SCHEMA}.lab_daily_orders", "gold"),
]
all_ok = True
for table, layer in tables:
    try:
        count = spark.table(table).count()
        print(f"  OK  [{layer:6}]  {table}  ->  {count:,} rows")
    except Exception as e:
        print(f"  FAIL [{layer:6}]  {table}  ->  {e}")
        all_ok = False
print()
print("All layers ready" if all_ok else "Some tables missing -- check task notebooks")

---

## Step 3 — Create the Databricks Workflow Job

### Step 3.1: Navigate to Workflows and create a new Job

Navigate to **Jobs & Pipelines** in the left sidebar, then click **Create Job**.

![Create Job](../../assets/images/training_2026/day3/6f524625f8464c29a2572b0a324333a5.webp)

---

### Step 3.2: Set the Job Name

Enter a **unique name** for your job: `<your_name>_lab_orders_pipeline`

![Job Name](../../assets/images/training_2026/day3/ae8ab8cafdcf4690b5d9665cbf058798.webp)

> **Note:** The job name does not need to be globally unique — each job is identified by a unique **Job ID** assigned automatically by Databricks.

![Job Details — ID & Creator](../../assets/images/training_2026/day3/460925af2b214f93a2252aed0ea86936.webp)

---

### Step 3.3: Configure Job Parameters

Add the following **job parameters** so that all tasks inherit the same configuration:

| Parameter | Value |
|-----------|-------|
| `catalog` | `retailhub_<your_name>` |
| `source_path` | `/Volumes/retailhub_<your_name>/default/datasets` |

![Job Parameters](../../assets/images/training_2026/day3/e24f8fbe50f642e0a28054972140dcb1.webp)

---

### Step 3.4: Add the First Task — `bronze_load`

Click **Add task** and configure:

| Field | Value |
|-------|-------|
| **Task name** | `bronze_load` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | `notebooks/labs/final_labs/tasks/lab_task_01_bronze` |
| **Compute** | Shared training cluster |

![Add Task](../../assets/images/training_2026/day3/8d2245e891b849e5966aae38a0e33c5f.webp)

![Task Configuration](../../assets/images/training_2026/day3/c89c69190d40463faae2b5186c78c271.webp)

Click **Create task** when done.

---

### Step 3.5: Add the Second Task — `silver_transform`

Click **Add task** and configure:

| Field | Value |
|-------|-------|
| **Task name** | `silver_transform` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | `notebooks/labs/final_labs/tasks/lab_task_02_silver` |
| **Depends on** | `bronze_load` |
| **Compute** | Shared training cluster |

> Setting **Depends on** ensures `silver_transform` starts only after `bronze_load` succeeds.

---

### Step 3.6: Add the Third Task — `gold_aggregation`

| Field | Value |
|-------|-------|
| **Task name** | `gold_aggregation` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | `notebooks/labs/final_labs/tasks/lab_task_03_gold` |
| **Depends on** | `silver_transform` |
| **Compute** | Shared training cluster |

Your completed pipeline DAG should look like this:

![Orders Pipeline](../../assets/images/training_2026/day3/119b8804bea74938aff23d816140bf4b.webp)

---

### Step 3.7: Configure a Scheduled Trigger

Navigate to the **Triggers** tab and add a **Scheduled** trigger:

| Setting | Value |
|---------|-------|
| Type | Scheduled |
| Cron expression | `0 0 6 * * ?` — every day at 06:00 UTC |
| Timezone | `Europe/Warsaw` |
| Status | **Paused** (leave paused for the lab) |

---

### Step 3.8: Run the Pipeline

Click **Run now** to manually trigger the job.

![Run Job](../../assets/images/training_2026/day3/73685979cbb243b3af4e01fdcadb1a32.webp)

> **Expected result:** All three tasks complete in sequence — `bronze_load` first, then `silver_transform`, then `gold_aggregation`.
> Click any task → **Output** tab to inspect the `dbutils.notebook.exit()` JSON returned by each notebook.
> If a task fails → use **Repair Run** to rerun only the failed task and its downstream tasks (upstream successful tasks are skipped → saves cost).

---

### Reference: DAB YAML Definition — `lab_orders_pipeline`

> The YAML below shows the **Databricks Asset Bundle** equivalent of what you configured in the UI above.

```yaml
resources:
  jobs:
    lab_orders_pipeline:
      name: lab_orders_pipeline
      trigger:
        pause_status: PAUSED
        periodic:
          interval: 1
          unit: DAYS
      tasks:
        - task_key: bronze_load
          notebook_task:
            notebook_path: /Workspace/.../notebooks/labs/final_labs/tasks/lab_task_01_bronze
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
        - task_key: silver_transform
          depends_on:
            - task_key: bronze_load
          notebook_task:
            notebook_path: /Workspace/.../notebooks/labs/final_labs/tasks/lab_task_02_silver
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
        - task_key: gold_aggregation
          depends_on:
            - task_key: silver_transform
          notebook_task:
            notebook_path: /Workspace/.../notebooks/labs/final_labs/tasks/lab_task_03_gold
            source: WORKSPACE
          existing_cluster_id: <CLUSTER_ID>
      parameters:
        - name: catalog
          default: retailhub_<your_name>
        - name: source_path
          default: /Volumes/retailhub_<your_name>/default/datasets
```

---

## Final checklist

- [ ] `lab_task_01_bronze` runs without error
- [ ] `lab_task_02_silver` runs without error
- [ ] `lab_task_03_gold` runs without error
- [ ] Smoke test (Step 2) shows OK for all 3 layers
- [ ] Job created: `bronze_load → silver_transform → gold_aggregation`
- [ ] Job parameters configured (`catalog`, `source_path`)
- [ ] Scheduled trigger configured (CRON `0 0 6 * * ?`, timezone `Europe/Warsaw`, status Paused)
- [ ] Manual run succeeded — all 3 tasks green in the Run History
- [ ] Inspected task output JSON in the Output tab

> ← [M04: Workflows & Automation](../modules/04_orchestration_jobs.ipynb) | [M00: Intro](../modules/00_intro.ipynb) →
